In [2]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)

import ray
import json
import math
import torch
from fastparquet import ParquetFile
from pathlib import Path
import webdataset as wds
from itertools import islice

In [ ]:
shard = Path("_shards_4")
shards_path = Path(f"/davinci-1/work/lbaroncelli/datacomp/{shard}")
tar_files = sorted([str(shards_path/s) for s in shards_path.glob("*.tar")])

In [ ]:
from PIL import Image
import imageio.v2 as imageio
import io

dataset = (
    wds.WebDataset(tar_files)
    .decode(
        wds.handle_extension(".jpg", lambda value: Image.fromarray(imageio.imread(io.BytesIO(value)))),
        wds.handle_extension(".json", lambda value: json.loads(value.decode("utf-8")).get("uid", "unknown")),
        wds.handle_extension(".txt", lambda value: value.decode("utf-8").strip()),
    )
    .to_tuple("jpg", "json", "txt")  # Extract image, uid, and caption
    .batched(16)
)

In [ ]:
batch = next(iter(dataset))
images, uids, captions = batch
print(len(images), len(uids), len(captions))

In [ ]:
from data_quality_pipeline.src.made.data_pipeline.model_hype import model_init
from data_quality_pipeline.src.made.data_pipeline.steps.specificity_filtering import specificity
from tokenizer import tokenize

ref_path = "/davinci-1/work/fdimatteo/hype_weights/reference.pt"
ref = torch.load(ref_path)
img_ref, txt_ref = ref["img"], ref["txt"]

model, trs = model_init(pretrained='/archive/SSD/home/fdimatteo/Progetti/fair_spoke_8/meru/hype/ckpt.pt')
curv = model.curvature.exp()
images_tensors = torch.stack([trs(im) for im in images])
txt_tensors = torch.stack(map(lambda x: tokenize(x) for x in captions))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
images_tensors = images_tensors.to(device)
txt_tensors = txt_tensors.to(device)

model = model.cuda()
model = model.eval()


with torch.no_grad():
    images_feat = model.encode_image(images_tensors)
    images_spec = specificity(img_ref = img_ref, txt_ref = txt_ref, image=images_feat, curv=curv)